# Agent Roles Library - LangChain Integration

This notebook demonstrates how to use the Agent Roles Library with **LangChain**.

## What You'll Learn
- Load role definitions from YAML
- Create LangChain LCEL chains from role definitions
- Bind tools to role-based agents
- Execute tasks with structured prompts
- Combine multiple roles in a pipeline

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd().parent))

from langchain_core.language_models.fake import FakeListLLM
from langchain_core.tools import tool

from loader import RoleRegistry
from models import RuntimeContext
from adapters.langchain_adapter import LangChainAdapter

## Step 1: Create an LLM and Tools

In [ ]:
# For demonstration, we use a FakeListLLM
# In production, replace with your actual LLM (e.g., ChatOpenAI, ChatAnthropic)
llm = FakeListLLM(
    responses=[
        "Analysis: The dataset shows strong seasonal patterns with Q4 peaks. Recommendation: Implement dynamic pricing.",
        "Story: Once upon a data point, trends emerged from chaos... The end.",
]
)

# Define a simple tool
@tool
def calculate_risk_score(probability: float, impact: float) -> float:
    """Calculate risk score from probability and impact."""
    return probability * impact

@tool
def summarize_text(text: str, max_words: int = 50) -> str:
    """Summarize text to specified word count."""
    words = text.split()
    return " ".join(words[:max_words]) + "..." if len(words) > max_words else text

tools = [calculate_risk_score, summarize_text]
print(f"Tools registered: {[t.name for t in tools]}")

## Step 2: Index the Role Registry

In [ ]:
registry = RoleRegistry()
registry.index()

print(f"Total roles loaded: {len(registry.list_roles())}")

# List data analysis roles
data_roles = [r for r in registry.list_roles() if r.category == "data_analysis"]
print("\nData analysis roles:")
for r in data_roles:
    print(f"  - {r.name}")

## Step 3: Create a LangChain Chain from a Role

In [ ]:
# Load the Data Scientist role
role = registry.get("data_scientist")

print(f"Role: {role.name}")
print(f"Description: {role.description[:100]}...")
print(f"Expertise: {', '.join(role.expertise[:3])}")
print(f"Recommended tools: {role.recommended_tools}")

In [ ]:
# Create runtime context with tools
context = RuntimeContext(
    llm=llm,
    tools=tools,
)

# Adapt to LangChain chain
adapter = LangChainAdapter(context)
chain = adapter.adapt(role)

print(f"Chain created for: {role.name}")
print(f"Chain type: {type(chain).__name__}")

In [ ]:
# Invoke the chain
result = chain.invoke({"input": "Analyze this sales dataset and calculate risk scores"})
print("Result:")
print(result)

## Step 4: Create Multiple Role-Based Chains

In [ ]:
# Create chains for different roles
roles_to_demo = ["data_scientist", "bi_analyst", "data_storyteller"]

chains = {}
for role_id in roles_to_demo:
    role = registry.get(role_id)
    if role:
        chain = adapter.adapt(role)
        chains[role_id] = (role, chain)
        print(f"Created chain: {role.name}")

In [ ]:
# Compare system prompts across roles
for role_id, (role, chain) in chains.items():
    print(f"\n{'='*50}")
    print(f"Role: {role.name}")
    print(f"{'='*50}")
    prompt = adapter.build_system_prompt(role)
    print(prompt[:300] + "...")

## Step 5: Pipeline Multiple Roles Together

In [ ]:
# Simulate a pipeline: Data Scientist -> BI Analyst -> Data Storyteller

# Step 1: Data Scientist analyzes
ds_role, ds_chain = chains["data_scientist"]
analysis = ds_chain.invoke({"input": "Analyze quarterly revenue trends"})
print("STEP 1 - Data Scientist Output:")
print(analysis[:200] + "\n")

# Step 2: BI Analyst creates dashboard concept
bi_role, bi_chain = chains["bi_analyst"]
dashboard = bi_chain.invoke({
    "input": f"Design a dashboard for this analysis: {analysis[:100]}",
    "chat_history": []
})
print("STEP 2 - BI Analyst Output:")
print(dashboard[:200] + "\n")

# Step 3: Data Storyteller crafts narrative
story_role, story_chain = chains["data_storyteller"]
story = story_chain.invoke({
    "input": f"Create an executive summary from this analysis: {analysis[:100]}",
})
print("STEP 3 - Data Storyteller Output:")
print(story[:200])

## Step 6: Search and Discover Roles

In [ ]:
# Search for creative roles
creative_roles = registry.search("creative")
print(f"Found {len(creative_roles)} creative roles:")
for r in creative_roles:
    print(f"  - {r.name} ({r.category})")

# Search for governance
gov_roles = registry.search("governance")
print(f"\nFound {len(gov_roles)} governance roles:")
for r in gov_roles:
    print(f"  - {r.name}")

## Summary

This notebook demonstrated:
1. Loading role definitions from YAML files
2. Creating LangChain LCEL chains from role definitions
3. Binding tools to role-based chains
4. Building multi-role pipelines for complex workflows
5. Discovering roles through registry search

The Agent Roles Library enables you to define agent personas once and adapt them
to LangChain's flexible composition patterns, creating powerful data analysis pipelines.